In [ ]:
import pandas as pd
import json
from pathlib import Path

# Use the actual paths copied from VS Code Explorer
csv_path = Path(
    "../data/planning_area_coverage_pressure.csv"
)

geojson_path = Path(
    "../data/MasterPlan2025PlanningAreaBoundaryNoSea.geojson"
)

# Load the 32-row BigQuery analysis
df = pd.read_csv(csv_path)

# Load the official URA planning boundaries
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson = json.load(f)

# Standardise planning-area names
df["planning_area"] = (
    df["planning_area"]
    .str.upper()
    .str.strip()
)

# Set output folder
output_dir = csv_path.parent / "visualisations"
output_dir.mkdir(parents=True, exist_ok=True)

print("Planning areas loaded:", len(df))
print("URA polygons loaded:", len(geojson["features"]))
print("Output folder:", output_dir)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:

def create_singapore_map(column, title, legend_title, filename):

    # Yellow-orange-red palette
    red_orange_scale = [
        [0.00, "#FFFFCC"],
        [0.20, "#FFEDA0"],
        [0.40, "#FEB24C"],
        [0.60, "#FD8D3C"],
        [0.80, "#F03B20"],
        [1.00, "#BD0026"]
    ]

    # All 55 URA planning areas
    all_areas = [
        feature["properties"]["PLN_AREA_N"]
        for feature in geojson["features"]
    ]

    fig = go.Figure()

    # Layer 1: Grey background for all planning areas
    fig.add_trace(
        go.Choropleth(
            geojson=geojson,
            locations=all_areas,
            featureidkey="properties.PLN_AREA_N",
            z=[1] * len(all_areas),
            colorscale=[
                [0, "#E8E8E8"],
                [1, "#E8E8E8"]
            ],
            showscale=False,
            marker_line_color="#A0A0A0",
            marker_line_width=0.6,
            hovertemplate=(
                "%{location}<br>"
                "Not represented in HDB-linked charging analysis"
                "<extra></extra>"
            ),
            name="URA Planning Areas"
        )
    )

    # Layer 2: Actual analysis values
    plot_df = df.dropna(subset=[column]).copy()

    custom_columns = [
        "total_carparks",
        "carparks_with_strong_ev_candidates",
        "matched_charging_points",
        "candidate_coverage_pct",
        "occupancy_pct",
        "not_available_pct"
    ]

    custom_data = plot_df[custom_columns].to_numpy()

    fig.add_trace(
        go.Choropleth(
            geojson=geojson,
            locations=plot_df["planning_area"],
            featureidkey="properties.PLN_AREA_N",
            z=plot_df[column],
            customdata=custom_data,
            colorscale=red_orange_scale,
            colorbar=dict(title=legend_title),
            marker_line_color="#A0A0A0",
            marker_line_width=0.6,
            hovertemplate=(
                "<b>%{location}</b><br>"
                "HDB carparks: %{customdata[0]}<br>"
                "Strong candidate carparks: %{customdata[1]}<br>"
                "Matched charging points: %{customdata[2]}<br>"
                "Candidate coverage: %{customdata[3]:.2f}%<br>"
                "Occupancy: %{customdata[4]:.2f}%<br>"
                "Not available: %{customdata[5]:.2f}%"
                "<extra></extra>"
            ),
            name=legend_title
        )
    )

    # Geographic appearance
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        projection_type="mercator",
        showland=True,
        landcolor="#F1F3F5",
        showocean=True,
        oceancolor="#DCEEF8",
        showlakes=True,
        lakecolor="#DCEEF8"
    )

    # Layout
    fig.update_layout(
        title=title,
        height=700,
        paper_bgcolor="#FFFFFF",
        margin=dict(l=10, r=10, t=60, b=10)
    )

    # Save interactive HTML
    output_file = output_dir / filename

    fig.write_html(
        str(output_file),
        include_plotlyjs=True,
        full_html=True
    )

    print("Saved:", output_file)

    # Display in notebook
    fig.show()

    return fig


In [ ]:
# Map 1 — Historical EV charging occupancy
create_singapore_map(
    column="occupancy_pct",
    title="EV Charging Occupancy by URA Planning Area",
    legend_title="Occupancy (%)",
    filename="Singapore_EV_Occupancy_Map.html"
)

# Map 2 — HDB EV charging candidate coverage
create_singapore_map(
    column="candidate_coverage_pct",
    title="HDB EV Charging Candidate Coverage by URA Planning Area",
    legend_title="Candidate Coverage (%)",
    filename="Singapore_EV_Coverage_Map.html"
)